# Baseline MLE-STAR × MLE-bench Lite (backbone pareado)

Este notebook roda o **grafo oficial do MLE-STAR** com um adaptador fino de provider (implementação
open-source do Google em [google/adk-samples](https://github.com/google/adk-samples), sample
`machine-learning-engineering`)
nas **22 competições configuradas do MLE-bench Lite** (as que excedem o limite de disco são
reportadas como puladas), na **mesma infra Colab** e com o **mesmo backbone**
Gemini 3.0 Flash Preview (`google/gemini-3-flash-preview` via OpenRouter) usado pelo Kaggle Agents.

**Por que isso existe:** a comparação da tese usa números transplantados do paper do MLE-STAR
(Gemini 2.0 Flash / 2.5 Pro, cluster 8×V100, 3 seeds). Re-rodar o MLE-STAR com o mesmo modelo e
mesma máquina elimina o confound de backbone/infra e produz uma comparação de custo *medida* —
os dois pontos mais frágeis para publicação.

**Requisitos**
1. `OPENROUTER_API_KEY` — em Colab, salve em *Secrets* (ícone de chave).
2. Secrets `KAGGLE_USERNAME` e `KAGGLE_KEY`.
3. **Aceite as regras de cada competição no site do Kaggle** (uma vez por conta), senão o
   `mlebench prepare` falha com 403.
4. GPU (L4 recomendada) e disco: competições >10 GB podem estourar o disco do Colab
   (o MLE-STAR copia os dados para cada branch de solução). Veja `SKIP_LARGE_GB` abaixo.

**Saídas:** `RESULTS_DIR/results_mlestar.json` (por competição: medalhas, above_median, score bruto,
tempo de execução, seed), `runtime_manifest.json`, `pip_freeze.txt` e tabela comparativa final no formato da
Table 5 da monografia.

**Referências:** Nam et al., *MLE-STAR: Machine Learning Engineering Agent via Search and Targeted
Refinement* (NeurIPS 2025, arXiv:2506.15692); Chan et al., *MLE-bench* (arXiv:2410.07095).

In [ ]:
# 1) Dependências
# torch/pandas/sklearn já vêm no Colab; instalamos o ADK, o grader do MLE-bench e libs
# que as soluções geradas costumam importar.
ADK_SAMPLES_COMMIT = "68989de5a041a0be2321bfdb9f7d657b148e0558"
MLEBENCH_COMMIT = "507f92e1138bb6e40dac5c6ee7a6758e6424bf97"
!pip -q install "google-adk[extensions]==1.36.1" "litellm==1.83.14" python-dotenv kaggle lightgbm xgboost catboost
!pip -q install "git+https://github.com/openai/mle-bench.git@{MLEBENCH_COMMIT}"
!test -d /content/adk-samples/.git || git clone https://github.com/google/adk-samples.git /content/adk-samples
!git -C /content/adk-samples fetch --depth 1 origin {ADK_SAMPLES_COMMIT}
!git -C /content/adk-samples checkout --detach {ADK_SAMPLES_COMMIT}

In [ ]:
# 2) Credenciais
import os
from pathlib import Path


def _load_secret(name: str) -> str | None:
    """Read a secret from Colab userdata or the environment."""
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name)


# As chaves ficam apenas em variáveis Python do orquestrador, nunca no ambiente herdado
# pelos scripts gerados.
openrouter_key = _load_secret("OPENROUTER_API_KEY")
assert openrouter_key, "Defina OPENROUTER_API_KEY (Colab Secrets ou env)"

# Kaggle (para mlebench prepare)
kaggle_user = _load_secret("KAGGLE_USERNAME")
kaggle_key = _load_secret("KAGGLE_KEY")
assert kaggle_user and kaggle_key, (
    "Defina KAGGLE_USERNAME e KAGGLE_KEY (Colab Secrets ou env)"
)
for secret_name in (
    "OPENROUTER_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY",
    "OPENAI_API_KEY", "GOOGLE_API_KEY",
):
    os.environ.pop(secret_name, None)
print("Credenciais OK")

In [ ]:
# 3) Configuração do experimento
import hashlib

# Backbone pareado com o Kaggle Agents (tese), usando o slug operacional do OpenRouter.
MODEL = "google/gemini-3-flash-preview"
LITELLM_MODEL = f"openrouter/{MODEL}"
SEARCH_MODEL = LITELLM_MODEL
SEARCH_TOOL = "openrouter:web_search"
RUN_PROVIDER_SMOKE_TEST = True  # testa modelo e busca antes da execução cara

# Protocolo. O paper do MLE-STAR usa 3 seeds; 1 seed = paridade com a run única da tese.
SEEDS = [42, 43, 44]

# Orçamentos (o paper usa limite de 24h/competição; ajuste ao seu tempo de Colab)
MAX_WALL_CLOCK_S = 24 * 3600  # limite soft; subprocessos síncronos podem ultrapassá-lo
EXEC_TIMEOUT_S = 3600         # timeout por script gerado (config.exec_timeout)

# Forma do grafo do MLE-STAR (defaults do sample ADK; valores do paper em comentário)
NUM_SOLUTIONS = 2             # paper: 2
NUM_MODEL_CANDIDATES = 4      # paper: 4
OUTER_LOOP_ROUND = 4          # paper: 4
INNER_LOOP_ROUND = 4          # paper: 4
ENSEMBLE_LOOP_ROUND = 5       # paper: 5
USE_DATA_LEAKAGE_CHECKER = True
USE_DATA_USAGE_CHECKER = True

# Competições >SKIP_LARGE_GB são puladas (o MLE-STAR copia os dados por branch de solução;
# siim-isic ~116 GB não cabe no disco padrão do Colab). None = tentar todas.
SKIP_LARGE_GB = 20.0

# Resultados persistem no Drive; o workspace e o cache continuam no disco rápido local.
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
        RESULTS_DIR = Path("/content/drive/MyDrive/kaggle-agents/mlestar_results")
    except ImportError:
        print("Google Drive indisponível; usando /content (não persistente).")
        RESULTS_DIR = Path("/content/mlestar_results")
else:
    RESULTS_DIR = Path("/content/mlestar_results")
WORK_ROOT = Path("/content/mlestar_work")
MLEBENCH_DATA_DIR = Path.home() / ".cache" / "mle-bench" / "data"
CLEANUP_AFTER_RUN = True      # limpa workspace por seed e dados após todas as seeds
FORCE_RERUN = False           # True re-executa competições já presentes no results_mlestar.json
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# MLE-bench Lite: 22 competições (task_type/lower alimentam os prompts do MLE-STAR)
COMPETITIONS = [
    {"id": "aerial-cactus-identification",                     "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 0.025},
    {"id": "aptos2019-blindness-detection",                    "task_type": "Image Classification", "metric": "quadratic_weighted_kappa",  "lower": False, "size_gb": 10.22},
    {"id": "dog-breed-identification",                         "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.75},
    {"id": "dogs-vs-cats-redux-kernels-edition",               "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.85},
    {"id": "histopathologic-cancer-detection",                 "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 7.76},
    {"id": "leaf-classification",                              "task_type": "Image Classification", "metric": "log_loss",                  "lower": True,  "size_gb": 0.036},
    {"id": "plant-pathology-2020-fgvc7",                       "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 0.8},
    {"id": "ranzcr-clip-catheter-line-classification",         "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 13.13},
    {"id": "siim-isic-melanoma-classification",                "task_type": "Image Classification", "metric": "auc",                       "lower": False, "size_gb": 116.16},
    {"id": "denoising-dirty-documents",                        "task_type": "Image to Image",       "metric": "rmse",                      "lower": True,  "size_gb": 0.06},
    {"id": "detecting-insults-in-social-commentary",           "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.002},
    {"id": "jigsaw-toxic-comment-classification-challenge",    "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.06},
    {"id": "random-acts-of-pizza",                             "task_type": "Text Classification",  "metric": "auc",                       "lower": False, "size_gb": 0.003},
    {"id": "spooky-author-identification",                     "task_type": "Text Classification",  "metric": "log_loss",                  "lower": True,  "size_gb": 0.002},
    {"id": "new-york-city-taxi-fare-prediction",               "task_type": "Tabular Regression",   "metric": "rmse",                      "lower": True,  "size_gb": 5.7},
    {"id": "nomad2018-predict-transparent-conductors",         "task_type": "Tabular Regression",   "metric": "rmsle",                     "lower": True,  "size_gb": 0.006},
    {"id": "tabular-playground-series-dec-2021",               "task_type": "Tabular Classification", "metric": "accuracy",                "lower": False, "size_gb": 0.7},
    {"id": "tabular-playground-series-may-2022",               "task_type": "Tabular Classification", "metric": "auc",                     "lower": False, "size_gb": 0.57},
    {"id": "mlsp-2013-birds",                                  "task_type": "Audio Classification", "metric": "auc",                       "lower": False, "size_gb": 0.585},
    {"id": "the-icml-2013-whale-challenge-right-whale-redux",  "task_type": "Audio Classification", "metric": "auc",                       "lower": False, "size_gb": 0.29},
    {"id": "text-normalization-challenge-english-language",    "task_type": "Sequence to Sequence", "metric": "accuracy",                  "lower": False, "size_gb": 0.01},
    {"id": "text-normalization-challenge-russian-language",    "task_type": "Sequence to Sequence", "metric": "accuracy",                  "lower": False, "size_gb": 0.01},
]
competition_signature = hashlib.sha256(
    ",".join(comp["id"] for comp in COMPETITIONS).encode()
).hexdigest()[:12]
disk_cap = "all" if SKIP_LARGE_GB is None else str(SKIP_LARGE_GB)
RUN_FINGERPRINT = (
    f"openrouter:{MODEL}:{ADK_SAMPLES_COMMIT[:12]}:{MLEBENCH_COMMIT[:12]}:"
    f"c{competition_signature}-s{'-'.join(map(str, SEEDS))}-"
    f"n{NUM_SOLUTIONS}-m{NUM_MODEL_CANDIDATES}-o{OUTER_LOOP_ROUND}-"
    f"i{INNER_LOOP_ROUND}-e{ENSEMBLE_LOOP_ROUND}-"
    f"leak{int(USE_DATA_LEAKAGE_CHECKER)}-usage{int(USE_DATA_USAGE_CHECKER)}-"
    f"exec{EXEC_TIMEOUT_S}-wall{MAX_WALL_CLOCK_S}-cap{disk_cap}"
)
print(f"{len(COMPETITIONS)} competições configuradas | modelo: {MODEL} | seeds: {SEEDS}")

In [ ]:
# 4) Importar o MLE-STAR (ORDEM IMPORTA)
# O grafo é construído no import. Por isso configuramos o objeto LiteLlm e substituímos
# a busca nativa do ADK antes de importar qualquer módulo do sample.
import glob
import importlib
import importlib.machinery
import importlib.metadata
import json
import platform
import subprocess
import sys
import time
import types as py_types

agent_py = glob.glob("/content/adk-samples/**/machine_learning_engineering/agent.py", recursive=True)
assert agent_py, "Sample machine-learning-engineering não encontrado no adk-samples"
SAMPLE_DIR = Path(agent_py[0]).parent.parent  # .../machine-learning-engineering
PACKAGE_DIR = SAMPLE_DIR / "machine_learning_engineering"
if str(SAMPLE_DIR) not in sys.path:
    sys.path.insert(0, str(SAMPLE_DIR))
print(f"MLE-STAR sample: {SAMPLE_DIR}")

# O __init__.py do sample atual importa o agente imediatamente (e tenta configurar Google
# Cloud). Criamos somente o package namespace para poder alterar CONFIG antes do grafo.
for module_name in tuple(sys.modules):
    if module_name == "machine_learning_engineering" or module_name.startswith(
        "machine_learning_engineering."
    ):
        del sys.modules[module_name]
package = py_types.ModuleType("machine_learning_engineering")
package.__file__ = str(PACKAGE_DIR / "__init__.py")
package.__package__ = "machine_learning_engineering"
package.__path__ = [str(PACKAGE_DIR)]
package.__spec__ = importlib.machinery.ModuleSpec(
    "machine_learning_engineering", loader=None, is_package=True
)
package.__spec__.submodule_search_locations = package.__path__
sys.modules["machine_learning_engineering"] = package

from google.adk.models.lite_llm import LiteLlm
from litellm import acompletion, completion


async def openrouter_web_search(query: str) -> dict:
    """Search the public web through OpenRouter and return grounded evidence."""
    response = await acompletion(
        model=SEARCH_MODEL,
        api_key=openrouter_key,
        messages=[
            {
                "role": "system",
                "content": (
                    "Act as a web-search tool for an ML engineering agent. Search current "
                    "public sources and return concise evidence, source URLs, and directly "
                    "useful implementation details."
                ),
            },
            {"role": "user", "content": query},
        ],
        tools=[
            {
                "type": SEARCH_TOOL,
                "parameters": {"engine": "native", "max_total_results": 10},
            }
        ],
        temperature=0.0,
        max_tokens=4096,
        timeout=180,
        num_retries=2,
    )
    message = response.choices[0].message
    payload = message.model_dump() if hasattr(message, "model_dump") else {}
    provider_fields = payload.get("provider_specific_fields") or {}
    annotations = payload.get("annotations") or provider_fields.get("annotations") or []
    return {"query": query, "result": message.content or "", "annotations": annotations}


# O GoogleSearchTool é específico do backend Gemini. O LiteLlm descartaria esse built-in;
# a callable abaixo vira uma FunctionTool normal e preserva o estágio Search-First.
google_search_module = importlib.import_module("google.adk.tools.google_search_tool")
google_search_module.google_search = openrouter_web_search

routed_model = LiteLlm(
    model=LITELLM_MODEL,
    api_key=openrouter_key,
    drop_params=True,
    timeout=300,
    num_retries=3,
)
# O root lê esta variável durante a construção; ele é sobrescrito pelo objeto acima logo
# após o import. Usar o slug Gemini aqui evita resolução prematura de provider.
os.environ["ROOT_AGENT_MODEL"] = "gemini-3-flash-preview"

from machine_learning_engineering.shared_libraries import config as mle_config

TASKS_DIR = SAMPLE_DIR / "machine_learning_engineering" / "tasks"

mle_config.CONFIG.agent_model = routed_model
mle_config.CONFIG.num_solutions = NUM_SOLUTIONS
mle_config.CONFIG.num_model_candidates = NUM_MODEL_CANDIDATES
mle_config.CONFIG.outer_loop_round = OUTER_LOOP_ROUND
mle_config.CONFIG.inner_loop_round = INNER_LOOP_ROUND
mle_config.CONFIG.ensemble_loop_round = ENSEMBLE_LOOP_ROUND
mle_config.CONFIG.use_data_leakage_checker = USE_DATA_LEAKAGE_CHECKER
mle_config.CONFIG.use_data_usage_checker = USE_DATA_USAGE_CHECKER
mle_config.CONFIG.exec_timeout = EXEC_TIMEOUT_S
mle_config.CONFIG.data_dir = str(TASKS_DIR) + "/"

# Só agora o import do agente constrói o grafo com OpenRouter em todos os subagentes.
from machine_learning_engineering.agent import root_agent  # noqa: E402

root_agent.model = routed_model


def walk_agents(agent):
    yield agent
    for child in getattr(agent, "sub_agents", None) or []:
        yield from walk_agents(child)


llm_agents = [agent for agent in walk_agents(root_agent) if hasattr(agent, "model")]
misrouted = [agent.name for agent in llm_agents if agent.model is not routed_model]
assert not misrouted, f"Agentes fora do OpenRouter: {misrouted}"

from machine_learning_engineering.shared_libraries import debug_util  # noqa: E402
from machine_learning_engineering.shared_libraries import code_util  # noqa: E402
from machine_learning_engineering.sub_agents.initialization import agent as init_agent  # noqa: E402

assert init_agent.google_search is openrouter_web_search
assert debug_util.google_search is openrouter_web_search

SENSITIVE_CHILD_ENV = {
    "OPENROUTER_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY",
    "OPENAI_API_KEY", "GOOGLE_API_KEY", "ANTHROPIC_API_KEY",
    "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN",
    "HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN", "WANDB_API_KEY", "GITHUB_TOKEN",
}


def run_python_code_isolated(
    code_text: str, run_cwd: str, py_filepath: str, exec_timeout: int
) -> dict:
    """Execute generated code without propagating orchestrator credentials."""
    started = time.time()
    output_path = Path(run_cwd) / py_filepath
    output_path.write_text(code_text, encoding="utf-8")
    child_home = Path(run_cwd) / ".generated_home"
    child_home.mkdir(parents=True, exist_ok=True)
    child_env = os.environ.copy()
    for secret_name in SENSITIVE_CHILD_ENV:
        child_env.pop(secret_name, None)
    child_env.update(
        HOME=str(child_home),
        XDG_CONFIG_HOME=str(child_home / ".config"),
        KAGGLE_CONFIG_DIR=str(child_home / ".kaggle"),
    )
    try:
        result = subprocess.run(
            [sys.executable, py_filepath],
            check=False, cwd=run_cwd, capture_output=True, text=True,
            timeout=exec_timeout, env=child_env,
        )
        returncode, stdout, stderr = result.returncode, result.stdout, result.stderr
    except Exception as exc:
        returncode, stdout, stderr = 1, "", str(exc)
    return {
        "returncode": returncode,
        "stdout": stdout,
        "stderr": stderr,
        "execution_time": time.time() - started,
    }


code_util.run_python_code = run_python_code_isolated

if RUN_PROVIDER_SMOKE_TEST:
    probe = completion(
        model=LITELLM_MODEL,
        api_key=openrouter_key,
        messages=[{"role": "user", "content": "Reply only with: ok"}],
        temperature=0.0,
        max_tokens=16,
        timeout=60,
        num_retries=1,
    )
    assert probe.choices[0].message.content, "OpenRouter respondeu sem conteúdo"
    search_probe = await openrouter_web_search(
        "Use web search and return the official scikit-learn homepage URL."
    )
    assert search_probe["result"], "A busca do OpenRouter respondeu sem conteúdo"

sample_commit = subprocess.run(
    ["git", "-C", str(SAMPLE_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert sample_commit == ADK_SAMPLES_COMMIT, (sample_commit, ADK_SAMPLES_COMMIT)
try:
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        check=False, capture_output=True, text=True,
    ).stdout.strip()
except FileNotFoundError:
    gpu_info = ""
runtime_manifest = {
    "provider": "openrouter",
    "model": MODEL,
    "litellm_model": LITELLM_MODEL,
    "search_model": SEARCH_MODEL,
    "search_tool": SEARCH_TOOL,
    "run_fingerprint": RUN_FINGERPRINT,
    "adk_samples_commit": sample_commit,
    "mlebench_commit": MLEBENCH_COMMIT,
    "google_adk_version": importlib.metadata.version("google-adk"),
    "litellm_version": importlib.metadata.version("litellm"),
    "python": sys.version,
    "platform": platform.platform(),
    "gpu": gpu_info or None,
    "generated_code_environment": "credentials_scrubbed_and_isolated_home",
    "protocol": {
        "seeds": SEEDS,
        "num_solutions": NUM_SOLUTIONS,
        "num_model_candidates": NUM_MODEL_CANDIDATES,
        "outer_loop_round": OUTER_LOOP_ROUND,
        "inner_loop_round": INNER_LOOP_ROUND,
        "ensemble_loop_round": ENSEMBLE_LOOP_ROUND,
        "use_data_leakage_checker": USE_DATA_LEAKAGE_CHECKER,
        "use_data_usage_checker": USE_DATA_USAGE_CHECKER,
        "max_wall_clock_s": MAX_WALL_CLOCK_S,
        "exec_timeout_s": EXEC_TIMEOUT_S,
        "skip_large_gb": SKIP_LARGE_GB,
    },
}
(RESULTS_DIR / "runtime_manifest.json").write_text(
    json.dumps(runtime_manifest, indent=2), encoding="utf-8"
)
freeze_process = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=False, capture_output=True, text=True,
)
if freeze_process.returncode == 0:
    pip_freeze = freeze_process.stdout
else:
    pip_freeze = "\n".join(sorted(
        f"{dist.metadata.get('Name', 'unknown')}=={dist.version}"
        for dist in importlib.metadata.distributions()
    )) + "\n"
(RESULTS_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")

print(f"Provider OK: {routed_model.model} | {len(llm_agents)} agentes LLM")
print("Busca Search-First OK:", SEARCH_MODEL)
print("Soluções paralelas:", mle_config.CONFIG.num_solutions)

In [ ]:
# 5) Helpers MLE-bench: preparar dados, montar task dir, grade
import json
import shutil
import subprocess


def ensure_prepared(comp_id: str) -> Path:
    """Roda `mlebench prepare -c <comp>` se necessário e retorna o dir public/."""
    public = MLEBENCH_DATA_DIR / comp_id / "prepared" / "public"
    if public.exists() and any(public.iterdir()):
        return public
    print(f"  mlebench prepare -c {comp_id} (pode demorar: download do Kaggle)")
    prepare_env = os.environ.copy()
    prepare_env.update(KAGGLE_USERNAME=kaggle_user, KAGGLE_KEY=kaggle_key)
    proc = subprocess.run(
        ["mlebench", "prepare", "-c", comp_id],
        capture_output=True, text=True, timeout=4 * 3600, env=prepare_env,
    )
    if not public.exists():
        raise RuntimeError(
            f"prepare falhou para {comp_id}:\n{proc.stdout[-1000:]}\n{proc.stderr[-1000:]}"
        )
    return public


def get_description(comp_id: str) -> str:
    """Descrição oficial da competição via registry do mlebench (com fallback)."""
    try:
        from mlebench.registry import registry

        comp = registry.set_data_dir(MLEBENCH_DATA_DIR).get_competition(comp_id)
        desc = getattr(comp, "description", "") or ""
        if desc:
            return desc
    except Exception as e:
        print(f"  registry description indisponível ({e}); usando fallback")
    for md in (MLEBENCH_DATA_DIR / comp_id).rglob("description*.md"):
        return md.read_text(encoding="utf-8", errors="ignore")
    return ""


def build_task_dir(comp: dict) -> Path:
    """Monta tasks/<comp>/ no layout do MLE-STAR: task_description.txt + dados públicos.

    O create_workspace do MLE-STAR copia TODO arquivo do task dir (exceto nomes contendo
    'answer'), então colocamos apenas o public/ do mlebench — o private/ (gabarito) fica fora.
    """
    comp_id = comp["id"]
    public = ensure_prepared(comp_id)
    task_dir = TASKS_DIR / comp_id
    if task_dir.exists():
        shutil.rmtree(task_dir)
    shutil.copytree(public, task_dir)

    files_listing = "\n".join(sorted(p.name for p in task_dir.iterdir()))
    description = get_description(comp_id)
    task_description = (
        f"# Task ({comp['task_type']})\n\n"
        f"{description}\n\n"
        f"# Metric\n{comp['metric']}\n\n"
        f"# Submission\nProduce a submission file exactly in the format of "
        f"sample_submission.csv provided with the data.\n\n"
        f"# Available data files\n{files_listing}\n"
    )
    (task_dir / "task_description.txt").write_text(task_description, encoding="utf-8")
    return task_dir


def grade_submission(comp_id: str, submission_path: Path) -> dict:
    """`mlebench grade-sample` -> dict com valid_submission/score/medals/above_median."""
    try:
        proc = subprocess.run(
            ["mlebench", "grade-sample", str(submission_path), comp_id],
            capture_output=True, text=True, timeout=300,
        )
        output = proc.stdout + proc.stderr
        start, end = output.find("{"), output.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(output[start:end])
        return {"valid_submission": False, "error": f"parse: {output[-400:]}"}
    except Exception as e:
        return {"valid_submission": False, "error": str(e)}


def cleanup_run(comp_id: str, workspace_dir: Path) -> None:
    """Remove artefatos volumosos de uma seed, preservando o download preparado."""
    for path in (TASKS_DIR / comp_id, workspace_dir):
        shutil.rmtree(path, ignore_errors=True)


def cleanup_data_cache(comp_id: str) -> None:
    """Remove os dados somente depois de concluir todas as seeds da competição."""
    shutil.rmtree(MLEBENCH_DATA_DIR / comp_id, ignore_errors=True)


print("Helpers prontos")

In [ ]:
# 6) Runner: executa o pipeline MLE-STAR de ponta a ponta para uma (competição, seed)
import asyncio
import time

from google.adk.runners import InMemoryRunner
from google.genai import types


async def run_pipeline(comp_id: str) -> str:
    """Envia a instrução ao frontdoor agent e consome o stream de eventos do ADK."""
    runner = InMemoryRunner(agent=root_agent, app_name="mle-star-baseline")
    session = await runner.session_service.create_session(
        app_name=runner.app_name, user_id="baseline"
    )
    content = types.Content(
        parts=[types.Part(text=f"execute the {comp_id} task")], role="user"
    )
    last_text, n_events = "", 0
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=content
    ):
        n_events += 1
        author = getattr(event, "author", "?")
        if n_events % 10 == 0:
            print(f"    [{time.strftime('%H:%M:%S')}] {n_events} eventos (último: {author})")
        try:
            if event.content and event.content.parts and event.content.parts[0].text:
                last_text = event.content.parts[0].text
        except Exception:
            pass
    return last_text


async def run_one(comp: dict, seed: int) -> dict:
    comp_id = comp["id"]
    workspace_dir = WORK_ROOT / f"seed{seed}"
    result = {
        "competition_id": comp_id,
        "seed": seed,
        "model": MODEL,
        "provider": "openrouter",
        "adk_samples_commit": ADK_SAMPLES_COMMIT,
        "run_fingerprint": RUN_FINGERPRINT,
        "task_type": comp["task_type"],
        "size_gb": comp["size_gb"],
        "valid_submission": False,
        "score": None,
        "gold_medal": False,
        "silver_medal": False,
        "bronze_medal": False,
        "any_medal": False,
        "above_median": False,
        "execution_time": 0.0,
        "error": None,
    }
    start = time.time()
    try:
        build_task_dir(comp)

        # Config runtime (lida pelo prepare_task a cada execução)
        mle_config.CONFIG.task_name = comp_id
        mle_config.CONFIG.task_type = comp["task_type"]
        mle_config.CONFIG.lower = comp["lower"]
        mle_config.CONFIG.seed = seed
        mle_config.CONFIG.workspace_dir = str(workspace_dir) + "/"

        try:
            await asyncio.wait_for(run_pipeline(comp_id), timeout=MAX_WALL_CLOCK_S)
        except asyncio.TimeoutError:
            result["error"] = f"wall-clock timeout ({MAX_WALL_CLOCK_S}s)"
            print(f"    TIMEOUT após {MAX_WALL_CLOCK_S}s — tentando grade parcial")

        # Submissão final do MLE-STAR: workspace/<task>/ensemble/final/submission.csv
        submission = workspace_dir / comp_id / "ensemble" / "final" / "submission.csv"
        if not submission.exists():
            candidates = list((workspace_dir / comp_id).rglob("submission.csv"))
            submission = candidates[0] if candidates else None

        if submission and submission.exists():
            saved = RESULTS_DIR / f"submission_{comp_id}_seed{seed}.csv"
            shutil.copy(submission, saved)
            grade = grade_submission(comp_id, submission)
            result.update(
                valid_submission=bool(grade.get("valid_submission")),
                score=grade.get("score"),
                gold_medal=bool(grade.get("gold_medal")),
                silver_medal=bool(grade.get("silver_medal")),
                bronze_medal=bool(grade.get("bronze_medal")),
                above_median=bool(grade.get("above_median")),
                grading_output=grade,
            )
            result["any_medal"] = bool(
                result["gold_medal"] or result["silver_medal"] or result["bronze_medal"]
            )
        else:
            result["error"] = result["error"] or "no submission produced"

        # Preserva o estado final (código gerado, scores) para o material suplementar
        final_state = workspace_dir / comp_id / "final_state.json"
        if final_state.exists():
            shutil.copy(final_state, RESULTS_DIR / f"final_state_{comp_id}_seed{seed}.json")
    except Exception as e:
        result["error"] = str(e)[:500]
    finally:
        result["execution_time"] = time.time() - start
        if CLEANUP_AFTER_RUN:
            cleanup_run(comp_id, workspace_dir / comp_id)
    return result


print("Runner pronto")

In [ ]:
# 7) Loop principal (retomável e agrupado por competição para reutilizar o download)
results_path = RESULTS_DIR / "results_mlestar.json"
loaded_results = json.loads(results_path.read_text()) if results_path.exists() else []
incompatible_results = [
    result for result in loaded_results
    if result.get("run_fingerprint") != RUN_FINGERPRINT
]
if incompatible_results:
    archive_path = RESULTS_DIR / (
        f"results_mlestar_incompatible_{time.strftime('%Y%m%d-%H%M%S')}.json"
    )
    archive_path.write_text(json.dumps(incompatible_results, indent=2, default=str))
    print(f"Resultados de outra configuração preservados em {archive_path}")
all_results = [
    result for result in loaded_results
    if result.get("run_fingerprint") == RUN_FINGERPRINT
]
if FORCE_RERUN and all_results:
    rerun_archive = RESULTS_DIR / (
        f"results_mlestar_forced_rerun_{time.strftime('%Y%m%d-%H%M%S')}.json"
    )
    rerun_archive.write_text(json.dumps(all_results, indent=2, default=str))
    print(f"Resultados anteriores preservados em {rerun_archive}")
    all_results = []
done = {
    (r["competition_id"], r.get("seed", 42))
    for r in all_results
    if not FORCE_RERUN
    and r.get("run_fingerprint") == RUN_FINGERPRINT
    and (r.get("valid_submission") is True or r.get("skipped") is True)
}

for i, comp in enumerate(COMPETITIONS, 1):
    oversized = SKIP_LARGE_GB is not None and comp["size_gb"] > SKIP_LARGE_GB
    for seed in SEEDS:
        key = (comp["id"], seed)
        header = f"[{i}/{len(COMPETITIONS)}] {comp['id']} (seed={seed})"
        if key in done:
            print(f"{header} — já executado, pulando")
            continue
        all_results = [
            previous for previous in all_results
            if (previous["competition_id"], previous.get("seed", 42)) != key
        ]
        if oversized:
            print(f"{header} — PULADO ({comp['size_gb']} GB > {SKIP_LARGE_GB} GB)")
            all_results.append({
                "competition_id": comp["id"], "seed": seed, "model": MODEL,
                "provider": "openrouter", "adk_samples_commit": ADK_SAMPLES_COMMIT,
                "run_fingerprint": RUN_FINGERPRINT,
                "task_type": comp["task_type"], "size_gb": comp["size_gb"],
                "skipped": True, "error": f"skipped: size > {SKIP_LARGE_GB} GB",
                "valid_submission": False, "any_medal": False, "above_median": False,
            })
            results_path.write_text(json.dumps(all_results, indent=2, default=str))
            continue
        print(f"\n{'=' * 70}\n{header}\n{'=' * 70}")
        result = await run_one(comp, seed)
        all_results.append(result)
        results_path.write_text(json.dumps(all_results, indent=2, default=str))
        medal = "🥇" if result["gold_medal"] else "🥈" if result["silver_medal"] else "🥉" if result["bronze_medal"] else "—"
        print(
            f"  -> valid={result['valid_submission']} score={result['score']} "
            f"medal={medal} above_median={result['above_median']} "
            f"({result['execution_time'] / 3600:.2f}h) erro={result['error']}"
        )
    if CLEANUP_AFTER_RUN:
        cleanup_data_cache(comp["id"])

print(f"\nConcluído. Resultados em {results_path}")

In [ ]:
# 8) Validar completude antes de publicar a tabela final
from collections import Counter

final_results = json.loads(results_path.read_text())
result_keys = [(r["competition_id"], r.get("seed", 42)) for r in final_results]
expected_keys = {(comp["id"], seed) for comp in COMPETITIONS for seed in SEEDS}
duplicate_keys = [key for key, count in Counter(result_keys).items() if count > 1]
missing_keys = sorted(expected_keys - set(result_keys))
unexpected_keys = sorted(set(result_keys) - expected_keys)
assert not duplicate_keys, f"Resultados duplicados: {duplicate_keys}"
assert not unexpected_keys, f"Resultados inesperados: {unexpected_keys}"
assert not missing_keys, (
    f"Execução parcial: faltam {len(missing_keys)} pares competição×seed. "
    "Retome a célula 7 antes de gerar a tabela final."
)
print(f"Completude OK: {len(result_keys)}/{len(expected_keys)} pares únicos")

In [ ]:
# 8) Agregação + tabela comparativa (formato da Table 5 da monografia)
import pandas as pd

df = pd.DataFrame(json.loads(results_path.read_text()))
skipped_mask = df.get("skipped", pd.Series(False, index=df.index)).fillna(False).astype(bool)
attempted = df[~skipped_mask]
n = len(attempted)


def pct(series) -> float:
    return 100.0 * series.fillna(False).astype(bool).sum() / max(n, 1)


mlestar_row = {
    "Sistema": f"MLE-STAR ({MODEL}, este notebook, n={n})",
    "Submissão válida %": round(pct(attempted["valid_submission"]), 1),
    "Acima da mediana %": round(pct(attempted["above_median"]), 1),
    "Medalhas %": round(pct(attempted["any_medal"]), 1),
    "Ouro %": round(pct(attempted.get("gold_medal", pd.Series(dtype=bool))), 1),
}

# Referências: tese (Kaggle Agents) e paper do MLE-STAR (Nam et al., 2025)
reference_rows = [
    {"Sistema": "AIDE (Gemini 2.0 Flash, paper)",        "Submissão válida %": 78.8,  "Acima da mediana %": 39.4, "Medalhas %": 25.8, "Ouro %": 12.1},
    {"Sistema": "MLE-STAR (Gemini 2.0 Flash, paper)",    "Submissão válida %": 95.5,  "Acima da mediana %": 63.6, "Medalhas %": 43.9, "Ouro %": 30.3},
    {"Sistema": "MLE-STAR (Gemini 2.5 Pro, paper)",      "Submissão válida %": 100.0, "Acima da mediana %": 83.3, "Medalhas %": 63.6, "Ouro %": 36.4},
    {"Sistema": "Kaggle Agents (Gemini 3.0 Flash, tese, protocolo legado)", "Submissão válida %": 100.0, "Acima da mediana %": 72.7, "Medalhas %": 59.1, "Ouro %": 27.3},
]

comparison = pd.DataFrame(reference_rows + [mlestar_row])
display(comparison)

print("\nPor competição:")
cols = ["competition_id", "seed", "valid_submission", "score", "any_medal", "above_median", "execution_time", "error"]
display(df[[c for c in cols if c in df.columns]])

total_h = attempted["execution_time"].fillna(0).sum() / 3600 if "execution_time" in attempted.columns else 0.0
print(f"\nTempo total de execução: {total_h:.1f} h GPU (custo L4 ~US$ {total_h * 0.67:.0f})")
comparison.to_csv(RESULTS_DIR / "comparison_table.csv", index=False)

In [ ]:
# 10) Dispersão entre seeds (não agregamos scores brutos de métricas diferentes)
seed_stats = (
    attempted.groupby("seed")
    .agg(
        n=("competition_id", "size"),
        valid_submission_rate=("valid_submission", "mean"),
        above_median_rate=("above_median", "mean"),
        any_medal_rate=("any_medal", "mean"),
        execution_time_h=("execution_time", lambda values: values.sum() / 3600),
    )
)
rate_columns = ["valid_submission_rate", "above_median_rate", "any_medal_rate"]
seed_stats[rate_columns] = 100 * seed_stats[rate_columns]
display(seed_stats.round(2))
display(seed_stats[rate_columns].agg(["mean", "std"]).round(2))

## Notas metodológicas (para a seção de experimentos do paper)

1. **Backbone pareado:** este baseline usa Gemini 3.0 Flash Preview
   (`google/gemini-3-flash-preview`) via OpenRouter, o mesmo backbone do Kaggle Agents,
   eliminando o confound de geração de modelo da comparação original (paper: 2.0 Flash / 2.5 Pro).
2. **Mesma infra:** uma GPU de Colab, mesmo disco/CPU — o custo por competição passa a ser *medido*
   na mesma máquina, e não estimado por proxy de preço de V100.
3. **Seeds:** este notebook já usa `SEEDS = [42, 43, 44]`, como o protocolo de 3 seeds do
   MLE-STAR; reporte média, dispersão e teste pareado por competição.
4. **Loops:** `NUM_MODEL_CANDIDATES=4` e os loops 4/4/5 já estão configurados para a
   paridade planejada. Registre qualquer redução de orçamento antes de comparar resultados.
5. **Busca/contaminação:** o `google_search` nativo do ADK não é transportado pelo LiteLLM.
   O notebook o substitui por uma FunctionTool que chama o server tool atual
   `openrouter:web_search` com o mesmo Gemini 3.0, preservando o estágio Search-First, mas não
   uma política de recuperação
   idêntica à do paper. Ela continua sem filtro anti-contaminação por competição; o Kaggle Agents
   aplica esse filtro. Reporte explicitamente essa diferença metodológica.
6. **Reprodutibilidade:** os commits do `adk-samples` e MLE-bench, Google ADK e LiteLLM estão
   fixados; `runtime_manifest.json` e `pip_freeze.txt` registram software e hardware. Resultados
   persistem no Drive e a retomada aceita apenas submissões válidas do mesmo fingerprint.
7. **Competições puladas por disco** (`SKIP_LARGE_GB`) devem ser reportadas explicitamente na
   tabela (ex.: siim-isic ~116 GB não cabe no disco do Colab porque o MLE-STAR copia os dados
   por branch de solução). O default de 20 GB a omite; use `None` apenas com disco suficiente.
8. **Limite de 24h:** `asyncio.wait_for` é um limite de orquestração, não um hard kill para
   ferramentas síncronas; registre o wall-clock real e qualquer excesso de até um timeout de componente.